In [12]:
import os

def find_missing_catROI(base_dir):
    missing_files = []

    # Walk recursively through base_dir
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.nii') and f.startswith('sub-') and 'T1w' in f:
                nii_path = os.path.join(root, f)


                # Extract subject ID (part before first underscore)
                subj_id = f.split('_')[0]

                # Look for 'label' folder sibling or one level up
                label_dir = os.path.join(root, 'label')
                if not os.path.isdir(label_dir):
                    # Try one level up
                    label_dir = os.path.join(os.path.dirname(root), 'label')

                catroi_found = False
                if os.path.isdir(label_dir):
                    for label_file in os.listdir(label_dir):

                        if (label_file.startswith(f'catROI_{subj_id}') and
                            'T1w' in label_file):
                            catroi_found = True
                            break

                if not catroi_found:
                    missing_files.append(nii_path)

    print("Files missing corresponding label/catROI files:")
    for f in missing_files:
        print(f)

def find_m0wp1sub_missing(base_dir):
    missing_files = []

    # Build a set of all sub- files with T1w for quick lookup
    sub_files_set = set()

    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.nii') and f.startswith('sub-') and 'T1w' in f:
                sub_files_set.add(f)

    # Now find all m0wp1sub-*.nii files with T1w
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.nii') and f.startswith('m0wp1sub-') and 'T1w' in f:
                # Extract subject id after 'm0wp1sub-'
                # e.g. m0wp1sub-01_something_T1w.nii -> sub-01
                # So subj_id = sub-XX
                remainder = f[len('m0wp1sub-'):]  # e.g. '01_something_T1w.nii'
                # Extract subject ID (first part before underscore or first few chars until non-digit)
                # Safer to parse like before: take up to first underscore
                if '_' in remainder:
                    sub_id_part = remainder.split('_')[0]  # '01'
                else:
                    sub_id_part = remainder.split('.')[0]

                # Compose the matching sub filename pattern, e.g. sub-01
                sub_prefix = f'sub-{sub_id_part}'

                # Now check if any file in sub_files_set starts with sub_prefix and contains T1w
                matched = any(sf.startswith(sub_prefix) and 'T1w' in sf for sf in sub_files_set)

                if not matched:
                    missing_files.append(os.path.join(root, f))

    print("\nm0wp1sub files missing corresponding sub- files:")
    for f in missing_files:
        print(f)

def find_m0wp1sub_missing(base_dir):
    missing_files = []

    for root, dirs, files in os.walk(base_dir):
        # Only look inside 'mri' folders
        if os.path.basename(root) != 'mri':
            continue

        for f in files:
            if f.endswith('.nii') and f.startswith('m0wp1sub-') and 'T1w' in f:
                # Extract subject ID after 'm0wp1sub-'
                remainder = f[len('m0wp1sub-'):]  # e.g. '01_something_T1w.nii'
                if '_' in remainder:
                    sub_id_part = remainder.split('_')[0]  # '01'
                else:
                    sub_id_part = remainder.split('.')[0]

                sub_prefix = f'sub-{sub_id_part}'

                # The parent directory of the 'mri' folder should contain the corresponding sub-*.nii files
                parent_dir = os.path.dirname(root)

                # Look for any file starting with sub_prefix and containing T1w in the parent directory
                matching_sub_found = False
                try:
                    for candidate in os.listdir(parent_dir):
                        if candidate.startswith(sub_prefix) and 'T1w' in candidate and candidate.endswith('.nii'):
                            matching_sub_found = True
                            break
                except FileNotFoundError:
                    # parent_dir may not exist, skip
                    pass

                if not matching_sub_found:
                    missing_files.append(os.path.join(root, f))

    print("\nm0wp1sub files missing corresponding sub- files:")
    for f in missing_files:
        print(f)

# Example usage:
base_dir = '../../Storage Repository/SRPBS/'
find_missing_catROI(base_dir)
find_m0wp1sub_missing(base_dir)

Files missing corresponding label/catROI files:
../../Storage Repository/SRPBS/data\sub-SRPBS0180\anat\sub-SRPBS0180_T1w.nii
../../Storage Repository/SRPBS/data\sub-SRPBS0181\anat\sub-SRPBS0181_T1w.nii
../../Storage Repository/SRPBS/data\sub-SRPBS0182\anat\sub-SRPBS0182_T1w.nii
../../Storage Repository/SRPBS/data\sub-SRPBS0183\anat\sub-SRPBS0183_T1w.nii
../../Storage Repository/SRPBS/data\sub-SRPBS0184\anat\sub-SRPBS0184_T1w.nii
../../Storage Repository/SRPBS/data\sub-SRPBS0185\anat\sub-SRPBS0185_T1w.nii

m0wp1sub files missing corresponding sub- files:
